# Práctica guiada — De formulario a experiencia de inferencia

En S5 ya construisteis un formulario Streamlit que recoge los 11 atributos y llama a un gateway basado en el contrato de S4. En S6 **no se rediseña el contrato ni se vuelve a implementar el formulario**. El objetivo es convertir esa pantalla funcional en una interfaz operable: mantiene el resultado al haber un rerun, hace visible lo que está ocurriendo y enseña la información justa para interpretar e investigar una predicción.

Trabajad en parejas. Completad los `TODO`, ejecutad las comprobaciones y conservad el notebook como evidencia para el taller. La implementación queda para la Clase 2.

## Entrega de la práctica

Al terminar debéis poder enseñar cuatro decisiones concretas:

1. qué permanece en la sesión, qué se cachea y qué no se retiene;
2. la máquina `idle → loading → success/error`;
3. un plano de pantalla con componentes Streamlit concretos;
4. mensajes seguros para confianza baja, lentitud y error.

No se entrega código de inferencia nuevo. S4 sigue siendo la única frontera de validación y predicción.

## 1. Comprobar el punto de partida de S5

Ejecutad la celda. Debe demostrar que partís del formulario y gateway de S5, no de una segunda implementación. Después anotad una limitación que observaríais al cambiar un widget o al enviar una segunda predicción.

In [2]:
from pathlib import Path

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'semana5/modules/05-streamlit-basic-model-ui').is_dir()
)
s5_app = repo_root / 'semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py'
source = s5_app.read_text(encoding='utf-8')

checks = {
    'reutiliza st.form': 'st.form(' in source,
    'recoge valores antes de inferir': 'collect_values' in source,
    'aún no usa session_state': 'st.session_state' not in source,
}
print('app de S5:', s5_app)
for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")
assert all(checks.values())

# TODO: escribid aquí una limitación observable de S5.
limitacion_s5 = "El resultado de la predicción no se guarda, por lo que puede desaparecer cuando Streamlit se ejecuta de nuevo"

print('Limitación observada:', limitacion_s5)

app de S5: c:\MLOps\icai-muiaap-operacion-de-modelos\semana5\modules\05-streamlit-basic-model-ui\solutions\02-first-streamlit\app.py
✓ reutiliza st.form
✓ recoge valores antes de inferir
✓ aún no usa session_state
Limitación observada: El resultado de la predicción no se guarda, por lo que puede desaparecer cuando Streamlit se ejecuta de nuevo


## 2. Decidir dónde vive cada cosa

Un rerun vuelve a ejecutar el script. Una variable local desaparece; `st.session_state` conserva información de **esa sesión**; `st.cache_resource` reutiliza recursos como un gateway o modelo. Un recurso cacheado global puede compartirse entre usuarios, así que debe ser seguro para concurrencia.

Completad la tabla y justificad cada decisión. No guardéis secretos ni las 11 features en la telemetría. Los valores del último envío solo se conservan para permitir un reintento y no se muestran ni se registran.

| Clave o recurso | Dónde vive | ¿Sobrevive a un rerun? | ¿Se limpia al pulsar «Limpiar»? | Motivo |
| --- | --- | --- | --- | --- |
| `last_state` | st.session_state | Si | Si | Conservar el estado y último resultado de la predicción |
| `last_values` | st.session_state | Si | Si | Guardar temporalmente los valores para poder reintentar una petición |
| `telemetry` | st.session_state | Si | No | Mantener las estadísticas de las peticiones durante la sesión |
| gateway / bundle | st.cache_resource | Si | No | 	Evitar cargar el modelo en cada rerun |
| respuesta de una petición | st.session_state | Si | Si | Conservar el resultado sin cachear respuestas individuales |

Referencias: [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) y [`st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource).

## 3. Diseñar la pantalla, no solo el estado

La interfaz debe tener cuatro zonas, en este orden:

1. **acción:** el formulario heredado de S5;
2. **operación:** qué está ocurriendo mientras se ejecuta;
3. **resultado:** categoría, confianza y latencia;
4. **trazabilidad:** versiones y `request_id`, visibles solo si se necesitan.

Elegid componentes que tengan una función. `st.columns` organiza, `st.status` comunica una operación, `st.metric` destaca un dato y `st.expander` aplaza el detalle técnico. No uséis una barra de progreso para la confianza: representa avance de una tarea, no certeza del modelo.

In [ ]:
# Sustituid cada TODO por el componente y la justificación que usaríais.
# Las claves describen zonas de la interfaz; no son nuevos contratos de ML.

ui_plan = {
    'formulario': {'component': 'st.form', 'reason': 'Reutilizar el formulario de la semana 5'},
    'estado_operacion': {'component': 'st.status', 'reason': 'Mostrar si la predicción está en ejecución'},
    'resumen_resultado': {'component': 'st.columns', 'reason': 'Mostrar la categoría, confianza y latencia en columnas'},
    'mensaje_interpretacion': {'component': 'st.warning', 'reason': 'Avisar cuando la confianza sea baja o la respuesta lenta'},
    'trazabilidad': {'component': 'st.expander', 'reason': 'Mostrar versiones y requests_id si son necesarios'},
    'separacion_telemetria': {'component': 'st.divider', 'reason': 'Separar la información de la telemetría'},
}

allowed_components = {
    'st.form', 'st.status', 'st.empty', 'st.columns', 'st.container',
    'st.metric', 'st.success', 'st.info', 'st.warning', 'st.error',
    'st.expander', 'st.divider', 'st.caption',
}
required_zones = {
    'formulario', 'estado_operacion', 'resumen_resultado',
    'mensaje_interpretacion', 'trazabilidad', 'separacion_telemetria',
}

assert set(ui_plan) == required_zones
for zone, decision in ui_plan.items():
    assert set(decision) == {'component', 'reason'}, zone

pending = [zone for zone, decision in ui_plan.items() if 'TODO' in decision.values()]
invalid = [
    f"{zone}: {decision['component']}"
    for zone, decision in ui_plan.items()
    if 'TODO' not in decision['component'] and decision['component'] not in allowed_components
]
print('Zonas pendientes:', pending or 'ninguna')
print('Componentes no permitidos:', invalid or 'ninguno')
print('Comprobación final: pending == [] and invalid == []')

Zonas pendientes: ninguna
Componentes no permitidos: ninguno
Comprobación final: pending == [] and invalid == []


### Plano mínimo esperado

```text
Título y contexto breve

[ formulario de S5 en dos columnas ]

[ estado de la operación ]

[ Categoría ]  [ Confianza ]  [ Latencia ]
mensaje interpretativo de confianza y latencia
▶ Trazabilidad: versiones y request_id

────────────────────────────────────
Telemetría agregada de la sesión
```

Podéis entregar este plano en el notebook o redibujarlo en draw.io. Justificad cualquier cambio respecto a esta estructura.

## 4. Máquina de estados y recuperación

Completad la tabla. Una respuesta válida con latencia superior a 300 ms sigue en `success`: añade un aviso técnico, no se convierte en error. La confianza es una señal para revisar, no un diagnóstico ni una garantía.


| Estado | Evento de entrada | Zona de pantalla que cambia | Qué se muestra | Acción permitida | Evidencia |
| --- | --- | --- | --- | --- | --- |
| `idle` | Inicio / limpiar | Operación | Formulario sin resultado | Enviar formulario | Estado idle |
| `loading` | Enviar formulario | Operación | Mensaje de ejecución | Esperar a que termine | Estado loading |
| `success` | Payload válido | Resultado | Categoría, confianza y latencia | Nueva predicción o limpiar | Resultado, versiones y request_id |
| `error` | Excepción controlada | Operación y mensajes | Explicación sencilla del error | Corregir, reintentar o limpiar | Error registrado en telemetría y request_id si está disponible |

Incluid una recuperación para entrada inválida, bundle no disponible y timeout. Indicad también qué no debe ver la persona: traceback, rutas internas, secretos o el payload completo.

| Problema | Qué se muestra | Cómo se recupera |
| --- | --- | --- |
| Entrada inválida | Mensaje para revisar los valores del formulario | Corregir los campos y volver a enviar |
| Bundle no disponible | Mensaje indicando que la predicción no está disponible | Reintentar cuando se restablezca el servicio |
| Timeout | Mensaje indicando que no se ha podido completar la petición | Reintentar con los últimos valores |

No se mostrarán al usuario tracebacks, rutas internas, claves secretas ni el payload completo de la petición

## 5. Escribir el copy de una interfaz responsable

Cada mensaje debe decir qué ha ocurrido, qué significa y qué puede hacer la persona. No debéis prometer que una predicción es correcta. Completad textos de una frase, sin terminología interna.

In [ ]:
ux_copy = {
    'low_confidence': 'La predicción tiene una confianza baja, revisa el resultado antes de usarlo',
    'slow_success': 'Se ha completado la predicción pero ha tardado más de lo esperado',
    'invalid_input': 'Se han introducido valores inválidos, revisa los campos y reintenta',
    'artifact_unavailable': 'No se puede realizar la predicción, inténtalo de nuevo más tarde',
}

assert set(ux_copy) == {
    'low_confidence', 'slow_success', 'invalid_input', 'artifact_unavailable'
}
for scenario, message in ux_copy.items():
    assert isinstance(message, str) and message.strip(), scenario

pending_copy = [scenario for scenario, message in ux_copy.items() if 'TODO' in message]
print('Mensajes pendientes:', pending_copy or 'ninguno')
print('Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?')

Mensajes pendientes: ninguno
Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?


#### Contrato de sesión

| Clave | Tipo | Inicialización | Ciclo de vida y limpieza |
| --- | --- | --- | --- |
| `last_state` | `PredictionState` | Estado `idle` al iniciar la sesión | Se actualiza con cada inferencia y vuelve a `idle` al limpiar |
| `last_values` | `dict[str, float]` o `None` | `None` al iniciar la sesión | Se guarda para permitir reintentos y se elimina al limpiar |
| `telemetry` | `Telemetry` | Contadores a cero y lista de latencias vacía | Se actualiza con cada petición y no se elimina al limpiar |

`PredictionState` contiene el estado de operación, mensaje, predicción, confianza y latencia.

`Telemetry` contiene el número total de peticiones, peticiones correctas, peticiones fallidas y latencias.

#### Plan de implementación 

| Requisito | Archivo / función |
| --- | --- |
| Inicializar y conservar el estado | `session.py` - `initialize_session_state()` |
| Limpiar sin perder telemetría | `session.py` - `clear_last_state()` |
| Validar los atributos de entrada | `gateway.py` - `DemoGateway.predict()` |
| Clasificar la confianza | `policies.py` - `classify_confidence()` |
| Clasificar la latencia | `policies.py` - `classify_latency()` |
| Preparar los resultados para la interfaz | `presentation.py` - `build_prediction_view()` |
| Traducir errores sin mostrar información interna | `presentation.py` - `to_user_facing_error()` |
| Gestionar los estados y registrar las predicciones | `controller.py` - `PredictionController.submit()` |
| Gestionar errores sin guardar el payload en telemetría | `controller.py` - `PredictionController.submit()` |
| Cachear el gateway | `app.py` - `get_cached_gateway()` |
| Mostrar los estados de la interfaz | `app.py` - `render_state()` |
| Integrar formulario, inferencia, reintentos y limpieza | `app.py` - `main()` |

## 6. Preparar el taller

Entregad el notebook con los TODO completados. Antes de cerrar, otra pareja debe poder responder estas preguntas leyendo vuestro trabajo:

- ¿Qué se mantiene al pulsar otro widget y qué se elimina con «Limpiar último resultado»?

Al pulsar otro widget se conservan `last_state`, `last_values` y `telemetry` mediante `st.session_state`. También se reutiliza el gateway mediante `st.cache_resource`
Al pulsar «Limpiar último resultado», se reinicia `last_state` a `idle` y se eliminan `last_values` y el resultado anterior. Se conservan `telemetry` y el gateway cacheado

- ¿Qué función cargará el gateway una sola vez y cuál será su límite?

Se utilizará una función como `get_gateway()` con `@st.cache_resource`, para cargar y reutilizar el gateway sin tener que cargar el modelo en cada rerun
La caché almacenará únicamente el recurso reutilizable, nunca los valores introducidos por el usuario ni las respuestas individuales

- ¿Qué componentes dibujará `render_state()` para `loading`, `success` y `error`?

`loading`: `st.status`
`success`: `st.success`
`error`: `st.error`

- ¿Qué ve una persona con confianza baja o una respuesta lenta?

Si la confianza es baja, se muestra un `st.warning` indicando que la predicción debe revisarse y que no garantiza un resultado correcto.Si la latencia supera los 300 ms, también aparece un aviso técnico, pero la predicción permanece en estado `success` si ha sido válida

- ¿Dónde aparecen el `request_id` y las versiones sin invadir el resultado?

Se muestran dentro de un `st.expander` llamado "Trazabilidad", debajo del resumen de la predicción.
De esta forma, el usuario puede consultar los detalles técnicos cuando los necesite, sin que interfieran con la categoría, confianza y latencia

En el taller se implementarán estas decisiones en `session.py`, `policies.py`, `presentation.py`, `controller.py` y `app.py`.

## Debrief

- ¿Qué limita a S5 aunque el formulario ya funcione?

El resultado puede desaparecer al producirse un rerun porque no se conserva en st.session_state

- ¿Por qué el gateway es un recurso cacheable y el resultado de una petición pertenece a una sesión?

Porque el gateway puede reutilizarse para varias predicciones, mientras que cada resultado corresponde a una petición concreta

- ¿Qué componente hace visible cada transición sin añadir una segunda lógica de inferencia?

st.status para la ejecución; st.success para el éxito; st.error para los fallos y st.info para informar cuando aún no hay ninguna predicción

- ¿Por qué categoría, confianza y latencia no tienen el mismo significado?

La categoría es la predicción del modelo, la confianza indica el grado de seguridad expresado por el modelo y la latencia mide cuánto ha tardado en responder.

- ¿Qué parte de este diseño se conserva al sustituir el gateway local por HTTP en S7?

Se mantienen los estados de la interfaz, los mensajes, la presentación del resultado y las políticas de UX. Cambia la implementación que realiza la petición al modelo.